In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from mne.viz import plot_topomap

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Transposed-Features ICA on Wavelet Power

## Scope

This notebook performs ICA decomposition on wavelet power data where
**time** serves as the observation (independent) axis and **all three
main dimensions — subjects, channels, and frequencies — are combined
into a single feature axis**:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)  — 4-D wavelet power
Reshape: (n_times,  n_subjects × n_channels × n_freqs)
         ── obs ──  ────────── features ──────────────
```

This is the **transpose** of the combined-features notebook
(`wavelet_ica_combined_features.ipynb`) which uses `(S×C×F, T)` with
S×C×F as observations and T as features.

## Why transpose?

Swapping observations and features fundamentally changes what PCA/ICA
discovers:

| | Combined Features (Approach 6) | **Transposed Features (this notebook)** |
|---|---|---|
| **Observations** | S × C × F (triplets) | T (time points) |
| **Features** | T (time) | S × C × F (triplets) |
| **Components are** | Temporal patterns (length T) | Spatial-spectral-subject patterns (length S×C×F) |
| **Scores are** | Per-triplet weights → `(S, C, F, K)` | Time courses → `(T, K)` |
| **What ICA finds** | Shared temporal dynamics | Shared spatial-spectral modes |
| **Key question** | *"Which time patterns are independent?"* | *"Which S×C×F activation patterns recur independently across time?"* |

In the combined-features approach, each ICA component IS a temporal
pattern, and the scores tell you how each subject–channel–frequency
triplet weights that pattern.  Here the roles are reversed: each ICA
component IS a spatial-spectral-subject pattern (a "mode" of brain
activation), and the scores tell you **when** that mode is active.

## What the decomposition finds

PCA followed by ICA discovers a small set of **spatial-spectral-subject
component patterns** (each of length S×C×F) that recur independently
across time.  The ICA source time course for each component reveals
**when** the mode is active.  The component pattern can be reshaped
back to `(S, C, F)` and decomposed into:

| Quantity | Shape | Interpretation |
|----------|-------|----------------|
| **ICA source** (time course) | `(T,)` | When the mode is active over the stimulus |
| **Subject loadings** | `(S,)` | Mean absolute mixing weight — which participants contribute most |
| **Channel loadings** | `(C,)` | Spatial topography (scalp map) |
| **Frequency loadings** | `(F,)` | Spectral profile of the mode |

## Analyses

1. Z-scoring and reshape
2. PCA dimensionality reduction + ICA decomposition
3. **(a)** Intersubject correlation matrix of ICA component mixing vectors
4. **(b)** Component time courses — ICA sources with per-subject variance
5. **(c)** Frequency × Time mean-loading heatmaps of mode activations
6. **(d)** Mean and variance of component mixing weights as scalp topomaps
7. **(e)** Per-subject loading bar plot for each component

## Configuration


In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ───────────────────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── Decomposition settings ────────────────────────────────────────────────────
N_COMPONENTS_PCA = 20  # number of PCA components to retain
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42  # reproducibility

# ── Storage directory ─────────────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "transposed_features"
    / "pca_ica"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading


In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.


In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.


In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(
    f"Shape      : {bb_data.shape}  (subjects \u00d7 channels \u00d7 freqs \u00d7 times)"
)
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Reshape (Transposed)

**Z-scoring** normalises each `(subject, channel, frequency)` time series
to zero mean and unit variance, exactly as in the combined-features
notebook.  This ensures PCA/ICA are not dominated by high-power features.

**Reshaping** then **transposes** the matrix so that **time is the
observation axis** and `S × C × F` is the feature axis:

```
(S, C, F, T)  →  flatten  →  (S×C×F, T)  →  transpose  →  (T, S×C×F)
                                                             obs  features
```

Each row of the resulting 2-D matrix is a single time point described
by all subject–channel–frequency combinations.  PCA/ICA will discover
**spatial-spectral-subject patterns** that recur independently across
time points — i.e. modes of coordinated brain activation.

### Contrast with Combined Features

In the combined-features notebook, PCA/ICA found **temporal patterns**
shared across observations (subject–channel–freq triplets).  Here, we
find **feature patterns** (which subject–channel–freq combinations are
co-activated) that recur across time.  The roles of observations and
features are swapped, yielding complementary perspectives on the same
data.


In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Reshape 4-D → 2-D → transpose:  (T, S×C×F)
n_features = n_subjects * n_channels * n_freqs
X_t = bb_z.reshape(-1, n_times).T  # (T, S*C*F)

print(f"Transposed matrix shape : {X_t.shape}")
print(f"  Observations (T)      : {X_t.shape[0]}")
print(f"  Features     (S\u00d7C\u00d7F) : {X_t.shape[1]}")
print(f"Column means \u2248 0 : {X_t.mean(axis=0).mean():.6f}")
print(f"Column stds  \u2248 1 : {X_t.std(axis=0).mean():.6f}")

---
## Step 2 — PCA Dimensionality Reduction + ICA Decomposition

PCA reduces the S×C×F feature space to `N_COMPONENTS_PCA` directions of
maximum variance.  FastICA then rotates these to maximise statistical
independence, yielding `N_COMPONENTS_ICA` independent components.

**Key difference from combined features:** here PCA/ICA operates on
`(T, S×C×F)` — **time points are observations**, so the components
found are patterns in the S×C×F feature space.  The ICA **sources**
(shape `(T, K)`) are the temporal activations of each mode, and the
ICA **mixing matrix** (shape `(S×C×F, K)`) describes which
subject–channel–frequency combinations participate in each mode.

**Results:**

| Object | Shape | Description |
|--------|-------|-------------|
| `ica_sources` | `(T, K)` | Temporal activation of each IC |
| `mixing_4d` | `(S, C, F, K)` | Full-space mixing weights reshaped to 4-D |
| `ica_full_mixing` | `(S×C×F, K)` | Full feature-space mixing matrix |


In [ ]:
# --- PCA ---
pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
pca_scores = pca.fit_transform(X_t)  # (T, K_pca)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance explained")
axes[0].set_title(f"PCA Scree Plot \u2014 {LABEL}")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance \u2014 {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"Top {N_COMPONENTS_PCA} components explain "
    f"{cumulative[-1] * 100:.1f}% of total variance."
)

# --- ICA ---
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
ica_sources = ica.fit_transform(pca_scores)  # (T, K_ica)

# Full feature-space mixing matrix: (S*C*F, K_ica)
# ICA mixing in PCA space: (K_pca, K_ica)
# PCA loadings: (K_pca, S*C*F)
# Full mixing = PCA_components^T @ ICA_mixing → (S*C*F, K_ica)
ica_full_mixing = pca.components_.T @ ica.mixing_  # (S*C*F, K_ica)

# Reshape mixing to 4-D for downstream analysis
mixing_4d = ica_full_mixing.reshape(
    n_subjects, n_channels, n_freqs, N_COMPONENTS_ICA
)  # (S, C, F, K)

print(f"ICA sources shape      : {ica_sources.shape}")
print(f"Full mixing shape      : {ica_full_mixing.shape}")
print(f"Mixing 4-D shape       : {mixing_4d.shape}")

---
## Analysis (a) — Intersubject Correlation Matrix of ICA Components

For each ICA component we compute a **subject × subject** Pearson
correlation matrix.  Each subject is represented by their
channel × frequency mixing-weight vector (shape `C × F`, flattened).

High off-diagonal correlations indicate that the mode has a consistent
spatial–spectral fingerprint across individuals, suggesting it is
stimulus-driven rather than noise.  If only a subset of subjects
correlate, the mode may reflect individual-difference factors.

### Interpretation difference from combined features

In the combined-features notebook, the ISC matrix uses **ICA scores**
(which capture how each triplet weights the temporal pattern).  Here
we use the **mixing matrix** (which captures how each triplet
participates in the spatial-spectral mode).  Both probe intersubject
consistency, but of different quantities.


In [ ]:
# Per-subject mixing vector for each IC: flatten (C, F) → (C*F,)
# mixing_4d: (S, C, F, K)  →  (S, C*F, K) after reshape
subject_cf_mixing = mixing_4d.reshape(
    n_subjects, n_channels * n_freqs, N_COMPONENTS_ICA
)  # (S, C*F, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Each subject's (C*F,) mixing vector for IC i
    corr_mat = np.corrcoef(subject_cf_mixing[:, :, i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Intersubject Correlation of IC Mixing Vectors \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "isc_component_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (b) — Component Time Courses (Mean ± Std Across Subjects)

The ICA sources `(T, K)` give the **global** temporal activation of
each mode — averaged implicitly across all S×C×F features.  To assess
**per-subject** variability, we reconstruct each subject's temporal
activation by projecting their z-scored data through the mixing weights:

For subject *s* and component *k*:

$$a_{s,k}(t) = \frac{1}{C \cdot F}\sum_{c,f} \text{mixing}_{s,c,f,k}\;\cdot\;z_{s,c,f}(t)$$

This weighted projection recovers the per-subject contribution to each
mode at every time point.  The **mean** across subjects (solid line)
with a shaded ± 1 standard deviation band reveals temporal segments
where the mode is consistently active versus highly variable across
individuals.

### Interpretation difference from combined features

In the combined-features notebook, the temporal pattern is the ICA
**component** itself (shape `(K, T)`), and per-subject activations are
derived from ICA scores.  Here, the temporal pattern is the ICA
**source** (shape `(T, K)`), and per-subject activations come from the
mixing matrix.  Both quantify "when is this mode active per subject",
but they arrive there from opposite directions.


In [ ]:
# Per-subject temporal activations for each IC
# mixing_4d: (S, C, F, K),  bb_z: (S, C, F, T)
# For each subject s, component k:
#   a_sk(t) = mean over (c, f) of [ mixing(s,c,f,k) * bb_z(s,c,f,t) ]
subject_temporal = np.einsum("scfk,scft->skt", mixing_4d, bb_z) / (
    n_channels * n_freqs
)  # (S, K, T)

mean_temporal = subject_temporal.mean(axis=0)  # (K, T)
std_temporal = subject_temporal.std(axis=0)  # (K, T)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, mean_temporal[i], lw=0.8, color="teal", label="mean")
    ax.fill_between(
        time,
        mean_temporal[i] - std_temporal[i],
        mean_temporal[i] + std_temporal[i],
        alpha=0.25,
        color="teal",
        label="\u00b1 1 std",
    )
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"Component {i + 1} \u2014 Temporal Activation", fontsize=10)
    if i == 0:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"ICA Mode Time Courses (mean \u00b1 std across subjects) \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_temporal_activations.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Analysis (c) — Frequency × Time Mean-Loading Heatmap

For each ICA mode, we compute the **mean loading per subject**
at each wavelet frequency and time point using the mixing matrix:

```
loading(k, f, t) = mean_{s,c}[ mixing_4d(s,c,f,k) × bb_z(s,c,f,t) ]
```

### Interpretation difference from combined features

In the combined-features notebook, the loading is computed with ICA
**scores** `scores_4d(s,c,f,k)`.  Here, it uses the ICA **mixing
matrix** `mixing_4d(s,c,f,k)`.  Both show which frequency–time
regions each IC is most active in, but the weights originate from
opposite sides of the decomposition.


In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
# Mean loading at each (freq, time): einsum over subjects and channels
ft_loading = np.einsum("scfk,scft->kft", mixing_4d, bb_z) / (
    n_subjects * n_channels
)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_loading[i]  # (F, T)
    vmin_s, vmax_s = np.percentile(data_i, 1), np.percentile(data_i, 99)
    ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="inferno",
        vmin=vmin_s,
        vmax=vmax_s,
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} \u2014 Freq \u00d7 Time Mean Loading", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Frequency \u00d7 Time Mean Loading per Mode \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_time_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


---
## Analysis (d) — Mean and Variance of Component Mixing Weights as Topomaps

Channel loadings are obtained from the **mixing matrix** by averaging
weights over frequencies for each subject, yielding a per-subject
channel-mixing matrix `(S, C, K)`.  We then compute:

- **Mean** across subjects → `(C, K)` — the average spatial distribution
  of each ICA mode.
- **Variance** across subjects → `(C, K)` — electrodes where the mode
  strength varies most between individuals.

Both are displayed as scalp topographic maps.

### Interpretation difference from combined features

In the combined-features notebook, topomaps come from ICA **scores**
(averaged over frequencies).  Here they come from the ICA **mixing
matrix** (averaged over frequencies).  Both represent channel-level
patterns, but scores and mixing weights have different statistical
properties: scores are per-observation weights, while mixing weights
define the component's spatial fingerprint.


In [ ]:
# Per-subject channel mixing: average over frequencies → (S, C, K)
ica_channel_mixing = mixing_4d.mean(axis=2)  # (S, C, K)

# Mean and variance across subjects
ica_ch_mean = ica_channel_mixing.mean(axis=0)  # (C, K)
ica_ch_var = ica_channel_mixing.var(axis=0)  # (C, K)

# Get MNE Info for topomap
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = min(6, N_COMPONENTS_ICA)

# --- Mean topomaps ---
_vlim_mean = np.percentile(np.abs(ica_ch_mean[:, :n_show]), 99)
fig_mean, axes_mean = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_mean = [axes_mean]

for i, ax in enumerate(axes_mean):
    im, _ = plot_topomap(
        ica_ch_mean[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        vlim=(-_vlim_mean, _vlim_mean),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_mean.suptitle(
    f"Mean Component Mixing (topomap) \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_mean[-1], label="mean mixing weight")
fig_mean.tight_layout()
if SAVE_PLOTS:
    fig_mean.savefig(PLOTS_DIR / "ica_topomap_mean.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

# --- Variance topomaps ---
_vmax_var = np.percentile(ica_ch_var[:, :n_show], 99)
fig_var, axes_var = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_var = [axes_var]

for i, ax in enumerate(axes_var):
    im, _ = plot_topomap(
        ica_ch_var[:, i],
        info,
        axes=ax,
        show=False,
        cmap="YlOrRd",
        vlim=(0, _vmax_var),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_var.suptitle(
    f"Variance of Component Mixing (topomap) \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_var[-1], label="variance")
fig_var.tight_layout()
if SAVE_PLOTS:
    fig_var.savefig(
        PLOTS_DIR / "ica_topomap_variance.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Analysis (e) — Per-Subject Mixing-Weight Bar Plot for Each Component

For each ICA component, compute the **mean absolute mixing weight**
across channels and frequencies per subject.  This scalar summarises
how strongly each participant's data participates in the mode.

Uniform subject weights indicate a stimulus-driven mode (shared brain
response); highly uneven weights may point to individual differences
(e.g. psilocybin response, attention state, or noise).

### Interpretation difference from combined features

In the combined-features notebook, bars show mean absolute **ICA score**
per subject.  Here they show mean absolute **mixing weight** per subject.
Scores and mixing weights are different quantities: scores are
per-observation (per-triplet) weights, while mixing weights define the
component's participation pattern.  Nevertheless, both answer the same
question: *"How strongly does each subject contribute to this component?"*


In [ ]:
# Subject mixing: mean |weight| over channels and frequencies
subject_mixing = np.abs(mixing_4d).mean(axis=(1, 2))  # (S, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.barh(
        range(n_subjects),
        subject_mixing[:, i],
        color="teal",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_xlabel("|mixing|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Subject")
fig.suptitle(
    f"Per-Subject Mixing Weight per Component \u2014 {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_subject_mixing.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Summary

### Key difference from combined features

The combined-features notebook (`wavelet_ica_combined_features.ipynb`)
treats `S×C×F` as **observations** and `T` as **features**.  ICA
components are temporal patterns; scores carry the S×C×F information.

This notebook does the opposite: `T` is **observations** and `S×C×F` is
**features**.  ICA sources are time courses; the mixing matrix carries
the S×C×F information.

| Aspect | Combined Features | Transposed Features (here) |
|--------|-------------------|----------------------------|
| Observations | S×C×F triplets | T time points |
| Features | T time | S×C×F triplets |
| Components represent | Temporal patterns | Spatial-spectral-subject modes |
| Time info lives in | ICA components (rows) | ICA sources (columns) |
| S×C×F info lives in | ICA scores (reshaped) | ICA mixing matrix (reshaped) |
| ISC uses | Score vectors per subject | Mixing vectors per subject |
| Topomaps from | Scores averaged over F | Mixing averaged over F |

Both approaches are valid and complementary.  The combined-features
approach is better at isolating **temporal dynamics** (e.g. event-locked
responses), while this transposed approach is better at isolating
**spatial-spectral modes** (e.g. network activation patterns).

### Variables available for further analysis

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X_t` | `(T, S×C×F)` | Z-scored transposed 2-D matrix |
| `pca` | — | Fitted PCA object |
| `pca_scores` | `(T, K_pca)` | PCA-transformed scores |
| `ica` | — | Fitted FastICA object |
| `ica_sources` | `(T, K_ica)` | ICA source time courses |
| `ica_full_mixing` | `(S×C×F, K_ica)` | Full feature-space mixing matrix |
| `mixing_4d` | `(S, C, F, K)` | Mixing matrix reshaped to 4-D |
| `subject_temporal` | `(S, K, T)` | Per-subject temporal activations |
| `ica_channel_mixing` | `(S, C, K)` | Per-subject channel mixing weights |

### Analyses implemented

| # | Analysis | Key finding |
|---|----------|-------------|
| (a) | Intersubject correlation matrix | Which modes have consistent S×C×F patterns across subjects |
| (b) | Temporal activations (mean ± std) | When each mode is active and how variable across subjects |
| (c) | Freq × Time mean loading | Mean IC loading at each wavelet frequency and time |
| (d) | Mean / variance topomaps | Spatial distribution and inter-individual variability |
| (e) | Per-subject mixing bars | Individual-level participation in each mode |

See `README.md` in this directory for the full analysis rationale,
alternative decomposition strategies, and ideas for future extensions.